# Global war and conflict — economic cost and impact (EDA)

Dataset: [khushikyad001/global-war-and-conflict-impact-dataset-19502024](https://www.kaggle.com/datasets/khushikyad001/global-war-and-conflict-impact-dataset-19502024).

Each row is a **synthetic conflict record** (parties A/B, conflict type, year span proxies, casualties, **Economic_Loss_USD_Billions**, weather/terrain, alliances, sanctions, refugees, outcome, geography). Treat **causal claims cautiously** — this is an exploratory panel for distributions and associations, not battlefield truth.

- **Primary cost field**: `Economic_Loss_USD_Billions` (continuous, USD billions).
- **Time**: `Year` (calendar year of the record; use aggregated time series, not event duration).
- **Plotly on Kaggle**: iframe renderer via `show_plotly` (same pattern as the air-travel notebook).

In [1]:
import os
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from pathlib import Path
from IPython.display import display, HTML
import plotly.colors as pc

PRISM = list(px.colors.qualitative.Prism)
px.defaults.template = "plotly_white"
px.defaults.color_discrete_sequence = PRISM
px.defaults.color_continuous_scale = [
    [i / max(len(PRISM) - 1, 1), c] for i, c in enumerate(PRISM)
]


def prism_rgba(color: str, alpha: float) -> str:
    c = color.strip()
    lc = c.lower()
    if lc.startswith("rgb"):
        r, g, b = (int(x) for x in pc.unlabel_rgb(c))
    else:
        h = c if c.startswith("#") else f"#{c}"
        r, g, b = pc.hex_to_rgb(h)
    return f"rgba({r},{g},{b},{alpha})"


_IS_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE")) or Path("/kaggle").exists()
if _IS_KAGGLE:
    pio.renderers.default = "iframe"


def show_plotly(fig):
    fig.update_layout(paper_bgcolor="white", plot_bgcolor="white", colorway=PRISM)
    if os.environ.get("PLOTLY_FORCE_HTML", "").lower() in ("1", "true", "yes"):
        display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False)))
    elif _IS_KAGGLE:
        fig.show()
    else:
        display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False)))

## Load data

On Kaggle, data usually lives under `/kaggle/input/...`. Locally, `kagglehub` downloads into the cache (or project `data/` if you copy the CSV).

In [2]:
import kagglehub

CSV_NAME = "global_conflicts_dataset.csv"


def resolve_csv_path() -> Path:
    kaggle_root = Path("/kaggle/input")
    if kaggle_root.exists():
        for p in kaggle_root.rglob(CSV_NAME):
            return p
    local_data = Path("data") / CSV_NAME
    if local_data.exists():
        return local_data
    path = kagglehub.dataset_download(
        "khushikyad001/global-war-and-conflict-impact-dataset-19502024"
    )
    p = Path(path) / CSV_NAME
    assert p.exists(), f"expected {CSV_NAME} under {path}"
    return p


csv_path = resolve_csv_path()
print("CSV:", csv_path)

df = pd.read_csv(csv_path, low_memory=False)
print(df.shape[0], "rows ×", df.shape[1], "cols")
df.head()

/Users/nicapotato/Documents/repo/project/KaggleNotebooks/eda/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CSV: /Users/nicapotato/.cache/kagglehub/datasets/khushikyad001/global-war-and-conflict-impact-dataset-19502024/versions/1/global_conflicts_dataset.csv
3000 rows × 30 cols


,Country_A,Country_B,Conflict_Type,Year,Duration_Days,Military_Deaths_A,Military_Deaths_B,Civilian_Deaths,Economic_Loss_USD_Billions,Temperature_Avg_C,...,Naval_Battles,Sanctions,Refugees_Millions,Ceasefire,Outcome,Latitude,Longitude,Climate_Zone,Resource_Dispute,UN_Involvement
0,France,France,Cold Conflict,2020,1829,33197,41305,50364,176.45,8.2,...,104,Yes,6.68,No,Victory_A,88.811,101.020,Temperate,Land,Yes
1,India,Japan,Cold Conflict,2013,1234,26773,10526,176846,435.83,24.8,...,87,Yes,14.51,Yes,Stalemate,55.878,78.502,Polar,Water,No
2,Israel,USA,Civil War,1970,1982,17256,7604,17280,154.50,17.3,...,9,No,1.63,No,Stalemate,29.263,144.680,Tropical,Water,No
3,Turkey,Australia,Proxy War,2021,1754,1745,33468,92279,273.20,12.3,...,15,No,7.23,No,Victory_A,-22.281,-147.397,Polar,Water,No
4,Australia,France,War,2012,753,29149,40672,72545,351.35,31.1,...,22,Yes,19.65,No,Victory_A,17.638,169.360,Temperate,Land,No


## Column types and measurement scales

We combine **pandas dtypes** with a short **semantic** label: nominal vs ordinal vs discrete vs continuous. Low-cardinality strings are treated as **nominal categorical** unless there is a clear order (here: none of the Yes/No fields are modeled as ordinal beyond binary).

In [3]:
NOMINAL = {
    "Country_A",
    "Country_B",
    "Conflict_Type",
    "Terrain_Type",
    "Alliance_A",
    "Alliance_B",
    "Weapons_Used",
    "Sanctions",
    "Ceasefire",
    "Outcome",
    "Climate_Zone",
    "Resource_Dispute",
    "UN_Involvement",
}

DISCRETE_INT = {
    "Year",
    "Duration_Days",
    "Military_Deaths_A",
    "Military_Deaths_B",
    "Civilian_Deaths",
    "Air_Strikes",
    "Naval_Battles",
}

CONTINUOUS = {
    "Economic_Loss_USD_Billions",
    "Temperature_Avg_C",
    "Rainfall_mm",
    "Population_A_Millions",
    "Population_B_Millions",
    "GDP_A_Billions",
    "GDP_B_Billions",
    "Refugees_Millions",
    "Latitude",
    "Longitude",
}

schema_rows = []
for col in df.columns:
    s = df[col]
    if col in NOMINAL:
        sem = "nominal categorical"
    elif col in DISCRETE_INT:
        sem = "integer (discrete / time index for Year)"
    elif col in CONTINUOUS:
        sem = "continuous (or treated as continuous)"
    else:
        sem = "(unclassified)"
    schema_rows.append(
        {
            "column": col,
            "dtype": str(s.dtype),
            "non_null": int(s.notna().sum()),
            "n_unique": int(s.nunique(dropna=True)),
            "semantic": sem,
        }
    )

schema = pd.DataFrame(schema_rows).sort_values("column").reset_index(drop=True)
display(schema)

num_cols = [c for c in df.columns if c in DISCRETE_INT | CONTINUOUS]
cat_cols = sorted(NOMINAL)
print("Numeric columns:", len(num_cols), "| Nominal columns:", len(cat_cols))

,column,dtype,non_null,n_unique,semantic
0,Air_Strikes,int64,3000,2223,integer (discrete / time index for Year)
1,Alliance_A,str,2017,2,nominal categorical
2,Alliance_B,str,2008,2,nominal categorical
3,Ceasefire,str,3000,2,nominal categorical
4,Civilian_Deaths,int64,3000,2983,integer (discrete / time index for Year)
5,Climate_Zone,str,3000,4,nominal categorical
6,Conflict_Type,str,3000,5,nominal categorical
7,Country_A,str,3000,15,nominal categorical
8,Country_B,str,3000,15,nominal categorical
9,Duration_Days,int64,3000,1565,integer (discrete / time index for Year)


Numeric columns: 17 | Nominal columns: 13


In [4]:
display(df.describe(include="all").T)
print("Missing values (top):")
display(df.isna().sum().sort_values(ascending=False).head(15))

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Country_A,3000,15,USA,224,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Country_B,3000,15,India,217,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Conflict_Type,3000,5,Skirmish,627,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Year,3000.0,NaN,NaN,NaN,1987.338,21.766006,1950.0,1968.0,1987.0,2007.0,2024.0
Duration_Days,3000.0,NaN,NaN,NaN,1003.734667,568.221869,2.0,529.0,999.0,1493.0,1999.0
Military_Deaths_A,3000.0,NaN,NaN,NaN,24423.385,14129.940099,105.0,12292.0,23889.0,36801.0,49995.0
Military_Deaths_B,3000.0,NaN,NaN,NaN,24959.623333,14345.873994,119.0,12768.25,24270.0,37424.5,49992.0
Civilian_Deaths,3000.0,NaN,NaN,NaN,100151.649333,57927.462774,68.0,49719.5,100106.5,149615.75,199972.0
Economic_Loss_USD_Billions,3000.0,NaN,NaN,NaN,253.68445,142.903418,0.23,132.905,257.58,374.8975,499.96
Temperature_Avg_C,3000.0,NaN,NaN,NaN,12.928667,13.094564,-10.0,1.7,13.0,24.2,35.0


Missing values (top):


Alliance_B           992
Alliance_A           983
Resource_Dispute     745
Country_A              0
Country_B              0
Climate_Zone           0
Longitude              0
Latitude               0
Outcome                0
Ceasefire              0
Refugees_Millions      0
Sanctions              0
Naval_Battles          0
Air_Strikes            0
Weapons_Used           0
dtype: int64

## Univariate — numeric distributions

Histograms for continuous and discrete cost/casualty proxies; `Year` as a discrete index.

In [5]:
KEY_NUM = [
    "Economic_Loss_USD_Billions",
    "Civilian_Deaths",
    "Military_Deaths_A",
    "Military_Deaths_B",
    "Refugees_Millions",
    "Duration_Days",
]

for col in KEY_NUM:
    fig = px.histogram(
        df,
        x=col,
        nbins=40,
        title=f"Distribution: {col}",
    )
    show_plotly(fig)

fig_y = px.histogram(df, x="Year", nbins=75, title="Records per Year (row counts)")
show_plotly(fig_y)

## Univariate — categorical frequencies

In [6]:
for col in ["Conflict_Type", "Outcome", "Terrain_Type", "Weapons_Used", "Climate_Zone"]:
    vc = df[col].value_counts(dropna=False)
    fig = px.bar(
        x=vc.index.astype(str),
        y=vc.values,
        labels={"x": col, "y": "count"},
        title=f"Counts: {col}",
    )
    show_plotly(fig)

## Boxplots — cost and casualties by conflict type and outcome

Compare spreads of **economic loss** and **civilian deaths** across `Conflict_Type` and `Outcome`.

In [7]:
for y in ["Economic_Loss_USD_Billions", "Civilian_Deaths", "Refugees_Millions"]:
    fig = px.box(
        df,
        x="Conflict_Type",
        y=y,
        color="Conflict_Type",
        points="outliers",
        title=f"{y} by Conflict_Type",
    )
    fig.update_layout(showlegend=False)
    show_plotly(fig)

fig2 = px.box(
    df,
    x="Outcome",
    y="Economic_Loss_USD_Billions",
    color="Outcome",
    points="outliers",
    title="Economic_Loss_USD_Billions by Outcome",
)
fig2.update_layout(showlegend=False)
show_plotly(fig2)

## Multivariate — correlation among numerics

Pearson correlation on complete cases for numeric columns only (pairwise deletion would differ slightly).

In [8]:
num_for_corr = df[num_cols].dropna()
cm = num_for_corr.corr(numeric_only=True)

fig = px.imshow(
    cm,
    text_auto=".2f",
    aspect="auto",
    color_continuous_scale="RdBu_r",
    zmin=-1,
    zmax=1,
    title="Pearson correlation (numeric columns)",
)
show_plotly(fig)

In [9]:
sample = df.sample(n=min(800, len(df)), random_state=42)
fig = px.scatter(
    sample,
    x="Civilian_Deaths",
    y="Economic_Loss_USD_Billions",
    color="Conflict_Type",
    size="Refugees_Millions",
    size_max=18,
    opacity=0.65,
    title="Economic loss vs civilian deaths (sample)",
    trendline="ols",
)
show_plotly(fig)

## Time series — aggregates by `Year`

Rows are not necessarily one conflict per year; we plot **mean / sum** of key fields by calendar year to see slow-moving patterns in this panel.

In [10]:
ts = (
    df.groupby("Year", as_index=False)
    .agg(
        n=("Country_A", "size"),
        economic_loss_mean=("Economic_Loss_USD_Billions", "mean"),
        economic_loss_sum=("Economic_Loss_USD_Billions", "sum"),
        civilian_deaths_mean=("Civilian_Deaths", "mean"),
        refugees_mean=("Refugees_Millions", "mean"),
    )
    .sort_values("Year")
)

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=ts["Year"],
        y=ts["economic_loss_mean"],
        name="Mean economic loss (USD B)",
        mode="lines+markers",
    )
)
fig.update_layout(
    title="Mean Economic_Loss_USD_Billions by Year",
    xaxis_title="Year",
    yaxis_title="Mean loss (USD billions)",
    hovermode="x unified",
)
show_plotly(fig)

fig2 = px.line(
    ts,
    x="Year",
    y="economic_loss_sum",
    markers=True,
    title="Sum of Economic_Loss_USD_Billions by Year (stacked rows)",
)
show_plotly(fig2)

fig3 = go.Figure()
fig3.add_trace(
    go.Scatter(
        x=ts["Year"],
        y=ts["civilian_deaths_mean"],
        name="Mean civilian deaths",
        mode="lines+markers",
        yaxis="y",
    )
)
fig3.add_trace(
    go.Scatter(
        x=ts["Year"],
        y=ts["refugees_mean"],
        name="Mean refugees (M)",
        mode="lines+markers",
        yaxis="y2",
    )
)
fig3.update_layout(
    title="Mean civilian deaths and mean refugees by Year (dual axis)",
    xaxis_title="Year",
    yaxis=dict(title="Mean civilian deaths", side="left"),
    yaxis2=dict(
        title="Mean refugees (millions)",
        overlaying="y",
        side="right",
    ),
    hovermode="x unified",
)
show_plotly(fig3)

In [11]:
ts_ct = (
    df.groupby(["Year", "Conflict_Type"], as_index=False)["Economic_Loss_USD_Billions"]
    .mean()
    .sort_values(["Year", "Conflict_Type"])
)
fig = px.line(
    ts_ct,
    x="Year",
    y="Economic_Loss_USD_Billions",
    color="Conflict_Type",
    markers=False,
    title="Mean economic loss by Year and Conflict_Type",
)
show_plotly(fig)